<div style="background:linear-gradient(135deg,#51247a 0%,#7a3fa0 100%);padding:22px 26px;border-radius:10px;margin-bottom:1.2rem;border-bottom:4px solid #f0a500;">
<div style="font-size:2.2rem;margin-bottom:6px;">📊</div>
<div style="color:white;font-size:1.5rem;font-weight:700;">Topic Explorer</div>
<div style="color:rgba(255,255,255,.8);font-size:.92rem;margin-top:4px;">Discover hidden themes using LDA topic modelling<br><a href="https://ladal.edu.au/topicmodels.html" style="color:#f0c060;font-size:.82rem;">&#x2192; See the full tutorial</a></div>
</div>


**What this tool does:** Uses Latent Dirichlet Allocation (LDA) to find recurring themes across your texts. Best results with medium-length texts (paragraph level).

**Steps:** Upload files &#x2192; Configure &#x2192; Run Analysis &#x2192; Download


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 1 &mdash; Set up the environment</b></div>


In [ ]:
suppressPackageStartupMessages(library(IRdisplay))
IRdisplay::display_html(paste0(
  "<style>",
  "  .jp-InputArea-editor { display: none !important; }",
  "  .jp-CodeCell .jp-Cell-inputWrapper { display: none !important; }",
  "  .show-code .jp-CodeCell .jp-Cell-inputWrapper { display: flex !important; }",
  "  .ladal-toolbar {",
  "    position: sticky; top: 0; z-index: 999;",
  "    background: #51247a; color: white;",
  "    padding: 8px 18px; border-radius: 0 0 8px 8px;",
  "    display: flex; align-items: center; gap: 14px;",
  "    font-family: sans-serif; font-size: .85rem;",
  "    box-shadow: 0 2px 8px rgba(81,36,122,.25);",
  "  }",
  "  .ladal-toolbar b { font-size: 1rem; }",
  "  .ladal-toolbar button {",
  "    background: rgba(255,255,255,.18);",
  "    border: 1px solid rgba(255,255,255,.4);",
  "    color: white; padding: 4px 12px; border-radius: 4px;",
  "    cursor: pointer; font-size: .8rem; font-weight: 600;",
  "  }",
  "  .ladal-toolbar button:hover { background: rgba(255,255,255,.32); }",
  "</style>",
  "<div class=\"ladal-toolbar\">",
  "  <b>&#x1F4D3; LADAL Interactive Notebook</b>",
  "  <button onclick=\"document.body.classList.toggle(&apos;show-code&apos;);",
  "    this.textContent=document.body.classList.contains(&apos;show-code&apos;)",
  "      ?&apos;Hide Code&apos;:&apos;Show Code&apos;\">Show Code</button>",
  "  <span style=\"opacity:.7;font-size:.78rem;\">",
  "    Code is hidden by default &mdash; click to toggle",
  "  </span>",
  "</div>"
))


In [ ]:
# ── Suppress startup messages & load display helpers ────────────────
suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))

# ── Colour-coded display helpers ────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:", bg, ";border-left:4px solid ", border,
    ";border-radius:6px;padding:11px 16px;margin:.6rem 0;",
    "font-family:sans-serif;font-size:.9rem;\">",
    if (nzchar(icon)) paste0("<b>", icon, "</b> ") else "",
    msg, "</div>"))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar ─────────────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    "<div style=\"font-family:sans-serif;font-size:.85rem;margin:.4rem 0;\">",
    "<span style=\"color:#51247a;font-weight:600;\">", label, "</span><br>",
    "<div style=\"background:#e8e4f0;border-radius:4px;height:10px;margin-top:4px;\">",
    "<div style=\"background:#51247a;width:", pct, "%;height:10px;",
    "border-radius:4px;transition:width .3s;\"></div></div>",
    "<span style=\"color:#888;font-size:.78rem;\">", pct, "%</span></div>"))
}

# ── Upload instructions ──────────────────────────────────────────────
upload_instructions <- function(folder="MyTexts", filetype="txt") {
  IRdisplay::display_html(paste0(
    "<div style=\"background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;",
    "padding:18px 22px;margin:.8rem 0;font-family:sans-serif;\">",
    "<b style=\"color:#51247a;font-size:1rem;\">&#x1F4C2; Upload your files</b><br><br>",
    "<ol style=\"margin:0;padding-left:1.4em;font-size:.9rem;line-height:1.8;\">",
    "<li>Find the <b>file browser panel</b> on the left side of the screen.</li>",
    "<li>Double-click the <b><code>", folder, "</code></b> folder to open it.</li>",
    "<li><b>Drag and drop</b> your <code>.", filetype, "</code> files into that folder.</li>",
    "<li>Come back here and click <b>Run Analysis</b> below.</li>",
    "</ol>",
    "<p style=\"margin:.6rem 0 0 0;font-size:.82rem;color:#888;\">",
    "Only <code>.", filetype, "</code> files are accepted. ",
    "You can upload multiple files at once.</p></div>"))
}

# ── Load plain-text files ────────────────────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in ", folder,
         ". Please upload files into the ", basename(folder), " folder.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts ───────────────────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save Excel ───────────────────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Environment ready. Scroll down to configure and run your analysis.")


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 2 &#x1F4C2; &mdash; Upload your texts</b></div>


In [ ]:
upload_instructions("MyTexts", "txt")

<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 3 &mdash; Configure and run</b></div>


In [ ]:
suppressPackageStartupMessages(library(ipywidgets))

w_ntopics <- ipywidgets::IntSlider(value=5, min=2, max=20, step=1,
               description="Number of topics:",
               style=list(description_width="160px"))
w_nterms  <- ipywidgets::IntSlider(value=10, min=5, max=30, step=5,
               description="Terms per topic:",
               style=list(description_width="160px"))
w_iter    <- ipywidgets::IntSlider(value=1000, min=200, max=3000, step=200,
               description="LDA iterations:",
               style=list(description_width="160px"))
w_stop    <- ipywidgets::Checkbox(value=TRUE,
               description="Remove English stopwords")
w_minf    <- ipywidgets::IntSlider(value=2, min=1, max=10, step=1,
               description="Min. doc. frequency:",
               style=list(description_width="160px"))
w_btn     <- ipywidgets::Button(description="  Run Analysis",
               button_style="primary", icon="bar-chart")
w_out     <- ipywidgets::Output()

ipywidgets::display(ipywidgets::VBox(list(
  ipywidgets::HTML("<b style=\"color:#51247a\">Configure topic model:</b>"),
  w_ntopics, w_nterms, w_iter, w_stop, w_minf, w_btn, w_out
)))

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      suppressPackageStartupMessages({
        library(topicmodels); library(quanteda)
        library(dplyr); library(tidyr); library(ggplot2); library(writexl)
      })
      .prog("Loading texts...", 8)
      texts <- load_texts("notebooks/MyTexts")
      ok(paste("Loaded", length(texts), "file(s)"))
      .prog("Building document-term matrix...", 20)
      toks <- tokens(corpus(texts),
                     remove_punct=TRUE, remove_numbers=TRUE,
                     remove_symbols=TRUE) |>
        tokens_tolower()
      if (w_stop$value)
        toks <- tokens_remove(toks, stopwords("en"), padding=FALSE)
      dfm_obj <- dfm(toks) |> dfm_trim(min_docfreq=w_minf$value)
      dfm_obj  <- dfm_obj[rowSums(dfm_obj) > 0, ]
      ok(paste0("DTM: ", ndoc(dfm_obj), " docs | ", nfeat(dfm_obj), " features"))
      if (ndoc(dfm_obj) < w_ntopics$value)
        stop("More topics than documents. Reduce the number of topics.")
      .prog(paste0("Fitting LDA (", w_ntopics$value, " topics, ",
                   w_iter$value, " iterations)..."), 35)
      dtm <- quanteda::convert(dfm_obj, to="topicmodels")
      lda <- topicmodels::LDA(dtm, k=w_ntopics$value, method="Gibbs",
               control=list(seed=42, iter=w_iter$value, burnin=100))
      .prog("Extracting terms...", 80)
      terms_df <- as.data.frame(topicmodels::terms(lda, w_nterms$value))
      colnames(terms_df) <- paste0("Topic_", seq_len(w_ntopics$value))
      terms_df$Rank <- seq_len(w_nterms$value)
      terms_df <- terms_df[, c("Rank", paste0("Topic_", seq_len(w_ntopics$value)))]
      print(terms_df)
      .prog("Plotting...", 88)
      gamma_df <- as.data.frame(topicmodels::posterior(lda)$topics)
      colnames(gamma_df) <- paste0("Topic_", seq_len(w_ntopics$value))
      gamma_df$document  <- rownames(gamma_df)
      p <- gamma_df |>
        tidyr::pivot_longer(-document, names_to="topic", values_to="prob") |>
        ggplot(aes(x=topic, y=prob, fill=topic)) +
        geom_col(show.legend=FALSE, width=.7) +
        facet_wrap(~document) +
        scale_fill_viridis_d(option="plasma") +
        coord_flip() + theme_bw(base_size=11) +
        labs(title="Topic distribution across documents", x=NULL, y="Probability")
      print(p)
      .prog("Saving...", 95)
      save_excel(terms_df, "topic_terms.xlsx")
      save_excel(gamma_df, "topic_documents.xlsx")
      ggsave("notebooks/MyOutput/topic_plot.png",
             bg="white", width=9, height=5, dpi=200)
      .prog("Done.", 100)
      ok("Saved topic_terms.xlsx, topic_documents.xlsx, topic_plot.png.")
    }, error=function(e) err(conditionMessage(e)))
  })
})


---

### Citation

Schweinberger, Martin. (2025). *LADAL Topic Explorer*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025topics,
  author = {Schweinberger, Martin},
  title  = {LADAL Topic Explorer},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()